# Insurance Premium Prediction — Google Colab Pipeline
## CPE232 Data Models - Hackathon 2

## Overview
This notebook is designed for Google Colab to participate in the Insurance Premium Prediction Hackathon. 
It includes:
- **Mounting Google Drive** for data access.
- **Data Preprocessing** (Feature Engineering, Missing Value Imputation).
- **Ensemble Modeling** (XGBoost + LightGBM + CatBoost).
- **GPU Acceleration** enabled for faster training.

## Dataset
- **Metric:** Mean Absolute Error (MAE).
- **Target:** `Premium Amount`.


## 1. Environment Setup & Import
Install necessary libraries if not present and import dependencies.

In [ ]:
!pip install catboost xgboost lightgbm --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 2. Mount Google Drive & Load Data
Ensure your data is uploaded to your Google Drive.

In [ ]:
from google.colab import drive
import os

# Mount Drive
try:
    drive.mount('/content/drive')
    print("Google Drive mounted.")
except:
    print("Drive mount failed or running locally.")

# Configuration - Adjust this path to your folder in Drive
# Example: "/content/drive/MyDrive/Hackathon2/cpe-232-insurance-premium-prediction"
BASE_PATH = "/content/drive/MyDrive/cpe232-datamodel-2025/hackathon/Hackathon2_Insurance-Premium-Prediction/cpe-232-insurance-premium-prediction"

train_path  = f"{BASE_PATH}/train.csv"
test_path   = f"{BASE_PATH}/test.csv"
sample_path = f"{BASE_PATH}/sample_submission.csv"

# Fallback to local upload if drive path doesn't exist
if not os.path.exists(train_path):
    print(f"Path not found: {BASE_PATH}")
    print("Please manually upload files to /content/ or fix the path.")
    BASE_PATH = "/content"
    train_path  = f"{BASE_PATH}/train.csv"
    test_path   = f"{BASE_PATH}/test.csv"
    sample_path = f"{BASE_PATH}/sample_submission.csv"

print(f"Loading data from: {train_path}")

try:
    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)
    sample_sub = pd.read_csv(sample_path)
    print(f"Train Shape: {train.shape}")
    print(f"Test Shape:  {test.shape}")
except FileNotFoundError:
    print("files not found! Please upload train.csv, test.csv, sample_submission.csv")

## 3. Exploratory Data Analysis (EDA)
Quick check on target distribution and correlations.

In [ ]:
# Target Distribution
plt.figure(figsize=(10, 5))
sns.histplot(train['Premium Amount'], bins=50, kde=True, color='green')
plt.title('Distribution of Premium Amount')
plt.show()

# Correlation Matrix
num_cols = train.select_dtypes(include=[np.number]).columns.drop(['id', 'Premium Amount'], errors='ignore')
corr = train[num_cols].corr()
plt.figure(figsize=(12, 10))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation Matrix")
plt.show()

## 4. Preprocessing & Feature Engineering
- Parse Dates.
- Impute Missing Values.
- Encode Categoricals.

In [ ]:
# Clean column names (remove leading/trailing spaces)
train.columns = train.columns.str.strip()
test.columns = test.columns.str.strip()

def preprocess_date(df):
    date_col = 'Policy Start Date'
    
    # Check if the column exists
    if date_col not in df.columns:
        print(f"Warning: '{date_col}' not found. Searching for alternatives...")
        found = False
        for col in df.columns:
            if 'start date' in col.lower() or 'policy date' in col.lower():
                print(f"Found alternative date column: '{col}'")
                date_col = col
                found = True
                break
        
        if not found:
            print(f"Error: Could not find date column. Available columns: {list(df.columns)}")
            return df
    
    # Keep original column name for consistency or rename? Let's use the found name but process it.
    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
    
    # Feature Engineering from Date
    df['Policy_Year'] = df[date_col].dt.year
    df['Policy_Month'] = df[date_col].dt.month
    df['Policy_Day'] = df[date_col].dt.day
    df['Policy_DayOfWeek'] = df[date_col].dt.dayofweek
    
    # Calculate Policy Duration in Days
    ref_date = df[date_col].max()
    df['Policy_Age_Days'] = (ref_date - df[date_col]).dt.days
    
    return df

def feature_engineering(df):
    # Text Length
    if 'Customer Feedback' in df.columns:
        df['Feedback_Len'] = df['Customer Feedback'].astype(str).apply(len)
    
    # --- Interaction Features (The Secret Sauce) ---
    # Safe division helper
    def safe_div(a, b):
        return a / (b + 1e-6)

    # 1. Income per Dependent (Financial stability per head)
    if 'Annual Income' in df.columns and 'Number of Dependents' in df.columns:
        df['Income_Per_Dependent'] = safe_div(df['Annual Income'], df['Number of Dependents'])
        
    # 2. Risk Indicators
    if 'Age' in df.columns and 'Health Score' in df.columns:
        # Age increases risk, Health Score reduces risk (usually)
        # Interaction identifying unhealthy old people (High Risk) vs Healthy young people (Low Risk)
        df['Age_Health_Interaction'] = df['Age'] * df['Health Score'] 
        df['Age_over_Health'] = safe_div(df['Age'], df['Health Score'])

    if 'Previous Claims' in df.columns and 'Vehicle Age' in df.columns:
        # Claims per year of vehicle life
        df['Claims_Density'] = safe_div(df['Previous Claims'], df['Vehicle Age'])

    # 3. Log Transforms for skewed money/count features
    if 'Annual Income' in df.columns:
        df['Log_Annual_Income'] = np.log1p(df['Annual Income'])

    return df

print("Processing Features...")
train = preprocess_date(train)
test = preprocess_date(test)

train = feature_engineering(train)
test = feature_engineering(test)

# Drop Columns that won't be used for training
drop_cols = ['id', 'Customer Feedback', 'Policy Start Date']
cols_to_drop = [c for c in drop_cols if c in train.columns]

X = train.drop(columns=['Premium Amount'] + cols_to_drop, errors='ignore')
y = train['Premium Amount']
X_test = test.drop(columns=cols_to_drop, errors='ignore')

# Imputation
num_features = X.select_dtypes(include=[np.number]).columns.tolist()
cat_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

# CLEANUP INF/NaN BEFORE IMPUTATION
# Replace Infinity with NaN so Imputer can handle it
X.replace([np.inf, -np.inf], np.nan, inplace=True)
X_test.replace([np.inf, -np.inf], np.nan, inplace=True)

# Numerical -> Median
if len(num_features) > 0:
    imputer_num = SimpleImputer(strategy='median')
    X[num_features] = imputer_num.fit_transform(X[num_features])
    X_test[num_features] = imputer_num.transform(X_test[num_features])

# Categorical -> Most Frequent
if len(cat_features) > 0:
    imputer_cat = SimpleImputer(strategy='most_frequent')
    X[cat_features] = imputer_cat.fit_transform(X[cat_features])
    X_test[cat_features] = imputer_cat.transform(X_test[cat_features])

# --- Robust Target Encoding ---
# (Using KFold to prevent leaks, strictly calculating means correctly)
from sklearn.model_selection import KFold

print("Performing Robust Target Encoding...")
kf_encoding = KFold(n_splits=5, shuffle=True, random_state=42)

for col in cat_features:
    new_col_name = col + '_target_enc'
    X[new_col_name] = np.nan
    
    for train_idx, val_idx in kf_encoding.split(X, y):
        # Join X and y for the fold
        X_train_fold, y_train_fold = X.iloc[train_idx], y.iloc[train_idx]
        
        # Calculate mean on training fold
        # We use a temporary DataFrame to group
        temp_df = X_train_fold[[col]].copy()
        temp_df['target'] = y_train_fold
        target_mean = temp_df.groupby(col)['target'].mean()
        
        # Map to validation fold
        X.loc[val_idx, new_col_name] = X.iloc[val_idx][col].map(target_mean)
        
    # Fill NaN in Train (unseen in fold) with global mean
    global_mean = y.mean()
    X[new_col_name].fillna(global_mean, inplace=True)
    
    # Map Test set using FULL Train data
    temp_full = X[[col]].copy()
    temp_full['target'] = y
    target_mean_full = temp_full.groupby(col)['target'].mean()
    
    X_test[new_col_name] = X_test[col].map(target_mean_full).fillna(global_mean)

# Label Encoding (Keep simplistic representations too)
for col in cat_features:
    le = LabelEncoder()
    full_data = pd.concat([X[col], X_test[col]], axis=0).astype(str)
    le.fit(full_data)
    X[col] = le.transform(X[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))
    
print("Preprocessing Complete. Features Engineered: ", X.shape[1])

## 5. GPU-Accelerated Model Training
We use XGBoost and CatBoost with GPU support.

In [ ]:
N_FOLDS = 10 
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

# Global placeholders for storage
oof_preds_xgb = np.zeros(len(X))
test_preds_xgb = np.zeros(len(X_test))

oof_preds_lgb = np.zeros(len(X))
test_preds_lgb = np.zeros(len(X_test))

oof_preds_cat = np.zeros(len(X))
test_preds_cat = np.zeros(len(X_test))

# --- 1. XGBoost (Ultra-Slow Learning) ---
print("\n========== Training XGBoost ==========")
xgb_params = {
    'n_estimators': 15000,          # Massive estimators
    'learning_rate': 0.005,         # Tiny learning rate
    'max_depth': 6,
    'min_child_weight': 20,         # High regularization (Conservativeness)
    'subsample': 0.60,
    'colsample_bytree': 0.60,
    'objective': 'reg:absoluteerror',
    'n_jobs': -1,
    'random_state': 42,
    'eval_metric': 'mae',
    'early_stopping_rounds': 800,   # Patience
    'tree_method': 'hist', 
    'device': 'cuda' 
}

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model = xgb.XGBRegressor(**xgb_params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    
    oof_preds_xgb[val_idx] = model.predict(X_val)
    test_preds_xgb += model.predict(X_test) / N_FOLDS
    if fold % 2 == 0: print(f"Fold {fold+1} MAE: {mean_absolute_error(y_val, oof_preds_xgb[val_idx]):.4f}")

mae_xgb = mean_absolute_error(y, oof_preds_xgb)
print(f"XGBoost Overall MAE: {mae_xgb:.4f}")


# --- 2. LightGBM (Ultra-Slow Learning) ---
print("\n========== Training LightGBM ==========")
lgb_params = {
    'n_estimators': 15000,
    'learning_rate': 0.005,
    'num_leaves': 25,               # Small leaves
    'min_child_samples': 80,        # Very robust
    'objective': 'mae',
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1,
    'subsample': 0.60,
    'colsample_bytree': 0.60
}

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model = lgb.LGBMRegressor(**lgb_params)
    
    callbacks = [lgb.early_stopping(stopping_rounds=800, verbose=False)]
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], eval_metric='mae', callbacks=callbacks)
    
    oof_preds_lgb[val_idx] = model.predict(X_val)
    test_preds_lgb += model.predict(X_test) / N_FOLDS
    if fold % 2 == 0: print(f"Fold {fold+1} MAE: {mean_absolute_error(y_val, oof_preds_lgb[val_idx]):.4f}")

mae_lgb = mean_absolute_error(y, oof_preds_lgb)
print(f"LightGBM Overall MAE: {mae_lgb:.4f}")


# --- 3. CatBoost (Ultra-Slow Learning) ---
print("\n========== Training CatBoost ==========")
cat_params = {
    'iterations': 15000,
    'learning_rate': 0.005,
    'depth': 6,
    'l2_leaf_reg': 10,              # Strong Reg
    'loss_function': 'MAE',
    'verbose': 0,
    'random_state': 42,
    'task_type': 'GPU',
    'devices': '0',
    'bootstrap_type': 'Bernoulli', 
    'subsample': 0.60
}

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model = CatBoostRegressor(**cat_params)
    model.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=800)
    
    oof_preds_cat[val_idx] = model.predict(X_val)
    test_preds_cat += model.predict(X_test) / N_FOLDS
    if fold % 2 == 0: print(f"Fold {fold+1} MAE: {mean_absolute_error(y_val, oof_preds_cat[val_idx]):.4f}")

mae_cat = mean_absolute_error(y, oof_preds_cat)
print(f"CatBoost Overall MAE: {mae_cat:.4f}")

## 6. Submission & Ensemble
Weighted average ensemble.

In [ ]:
# Use scipy.optimize to find exact best weights
from scipy.optimize import minimize

def minimize_mae(weights):
    final_pred = (weights[0] * oof_preds_xgb) + (weights[1] * oof_preds_lgb) + (weights[2] * oof_preds_cat)
    return mean_absolute_error(y, final_pred)

print("Optimizing Weights...")
init_weights = [0.33, 0.33, 0.33]
# Constraints: Sum of weights = 1
cons = ({'type': 'eq', 'fun': lambda w: 1 - sum(w)})
# Bounds: Each weight between 0 and 1
bounds = [(0.0, 1.0)] * 3

res = minimize(minimize_mae, init_weights, method='SLSQP', bounds=bounds, constraints=cons)
w_xgb, w_lgb, w_cat = res.x

print(f"Optimized Weights -> XGB: {w_xgb:.4f}, LGB: {w_lgb:.4f}, Cat: {w_cat:.4f}")

final_oof_preds = (w_xgb * oof_preds_xgb) + (w_lgb * oof_preds_lgb) + (w_cat * oof_preds_cat)
ensemble_mae = mean_absolute_error(y, final_oof_preds)
print(f"Optimized Ensemble OOF MAE: {ensemble_mae:.4f}")

# Final Test Predictions with Optimized Weights
final_preds = (w_xgb * test_preds_xgb) + (w_lgb * test_preds_lgb) + (w_cat * test_preds_cat)

submission = pd.DataFrame({
    'id': sample_sub['id'],
    'Premium Amount': final_preds
})

submission.to_csv('submission.csv', index=False)
print("Saved submission.csv with Optimized Weights")
submission.head()